In [1]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("jessemostipak/hotel-booking-demand")

# print("Path to dataset files:", path)

100%|█████████████████████████████████████████████████████████████████████████████| 1.25M/1.25M [00:00<00:00, 12.9MB/s]

Extracting files...
Path to dataset files: /home/chris/.cache/kagglehub/datasets/jessemostipak/hotel-booking-demand/versions/1


In [101]:
import pandas as pd
import numpy as np
df = pd.read_csv(f'{path}/hotel_bookings.csv')

In [102]:
with pd.option_context('display.max_columns', None):
    display(df.head())

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [103]:
df = df.drop(columns = ['reservation_status', 'reservation_status_date'])

In [104]:
for col in ['agent', 'company']:
    df[f'has_{col}'] = df[col].notna().astype(int)
    df[col] = df[col].apply(lambda v: 'none' if pd.isna(v) else str(int(v)))

In [105]:
with pd.option_context('display.max_columns', None):
    display(df.head())

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,has_agent,has_company
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,none,none,0,Transient,0.0,0,0,0,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,none,none,0,Transient,0.0,0,0,0,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,none,none,0,Transient,75.0,0,0,0,0
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304,none,0,Transient,75.0,0,0,1,0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240,none,0,Transient,98.0,0,1,1,0


In [106]:
df['children'] = df['children'].fillna(0).astype(int)
df['country'] = df['country'].fillna('unknown')

In [107]:
# no nulls after this point

In [108]:
cols = ['reserved_room_type', 'assigned_room_type']
df['got_room'] = np.where(df['reserved_room_type'] == df['assigned_room_type'], 1, 0)

In [109]:
df = df.drop(columns = cols)

In [110]:
import calendar
months = {m: i for i, m in enumerate(calendar.month_name) if m}

df['arrival_date'] = pd.to_datetime(dict(
    year=df['arrival_date_year'],
    month=df['arrival_date_month'].map(months),
    day=df['arrival_date_day_of_month'],
))

df['arrival_month'] = df['arrival_date'].dt.month
df['arrival_dow']   = df['arrival_date'].dt.dayofweek
df['is_weekend']    = (df['arrival_dow'] >= 5).astype(int)

df['month_sin'] = np.sin(2 * np.pi * df['arrival_month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['arrival_month'] / 12)

In [111]:
df = df.drop(columns=['arrival_date_year', 'arrival_date_month',
                      'arrival_date_week_number', 'arrival_date_day_of_month'])

In [112]:
from sklearn.preprocessing import OneHotEncoder

special_encoder = OneHotEncoder(handle_unknown='ignore', min_frequency=300, sparse_output=False)
special_encoder.set_output(transform='pandas')
special_cols = ['country', 'agent', 'company']

encoded_cols = special_encoder.fit_transform(df[special_cols])

df = pd.concat([df.drop(columns=special_cols), encoded_cols], axis=1)



In [113]:
# df['country'].value_counts().loc[lambda x: x > 100]
# df.head()
encoder = OneHotEncoder(handle_unknown = 'ignore', sparse_output=False)
encoder.set_output(transform='pandas')
cols = ['hotel', 'meal', 'distribution_channel', 'market_segment', 'deposit_type', 'customer_type', 'arrival_dow']

encoded_cols = encoder.fit_transform(df[cols])

df = pd.concat([df.drop(columns = cols), encoded_cols], axis = 1)
df = df.drop(columns = 'arrival_month')

In [114]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numeric_cols = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights' ,'adults' ,
                'children', 'babies', 'previous_cancellations', 'previous_bookings_not_canceled' , 'booking_changes',
                'days_in_waiting_list', 'adr' ,'total_of_special_requests' ,'required_car_parking_spaces']



In [126]:
from sklearn.model_selection import train_test_split
df = df.sort_values('arrival_date').reset_index(drop=True)
X = df.drop(columns = 'is_canceled')
y = df['is_canceled']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state =42,
    shuffle = False)
X_train = X_train.drop(columns = 'arrival_date')
X_test = X_test.drop(columns = 'arrival_date')


In [127]:
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train_scaled[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test_scaled[numeric_cols])

In [128]:
X_train_scaled.head()


,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,has_agent,has_company,got_room,is_weekend,month_sin,month_cos,country_AGO,country_AUS,country_AUT,country_BEL,country_BRA,country_CHE,country_CHN,country_CN,country_DEU,country_DNK,country_ESP,country_FIN,country_FRA,country_GBR,country_IRL,country_ISR,country_ITA,country_NLD,country_NOR,country_POL,country_PRT,country_ROU,country_RUS,country_SWE,country_USA,country_unknown,country_infrequent_sklearn,agent_1,agent_11,agent_119,agent_12,agent_134,agent_14,agent_15,agent_171,agent_177,agent_19,agent_196,agent_20,agent_21,agent_22,agent_229,agent_240,agent_241,agent_242,agent_243,agent_250,agent_26,agent_27,agent_273,agent_28,agent_29,agent_298,agent_3,agent_30,agent_314,agent_37,agent_40,agent_5,agent_56,agent_58,agent_6,agent_7,agent_8,agent_83,agent_85,agent_86,agent_9,agent_96,agent_none,agent_infrequent_sklearn,company_223,company_40,company_none,company_infrequent_sklearn,hotel_City Hotel,hotel_Resort Hotel,meal_BB,meal_FB,meal_HB,meal_SC,meal_Undefined,distribution_channel_Corporate,distribution_channel_Direct,distribution_channel_GDS,distribution_channel_TA/TO,distribution_channel_Undefined,market_segment_Aviation,market_segment_Complementary,market_segment_Corporate,market_segment_Direct,market_segment_Groups,market_segment_Offline TA/TO,market_segment_Online TA,market_segment_Undefined,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Contract,customer_type_Group,customer_type_Transient,customer_type_Transient-Party,arrival_dow_0,arrival_dow_1,arrival_dow_2,arrival_dow_3,arrival_dow_4,arrival_dow_5,arrival_dow_6
0,2.347806,-0.903659,-1.277636,0.267240,-0.245816,-0.081701,0,-0.112988,-0.09651,4.351915,-0.146045,-2.006196,-0.26026,-0.691011,0,0,1,0,-0.5,-0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.282557,-0.903659,0.804806,0.267240,-0.245816,-0.081701,0,-0.112988,-0.09651,-0.336686,-0.146045,-0.666097,-0.26026,-0.691011,1,0,1,0,-0.5,-0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,1.535092,-0.903659,-0.236415,-1.394628,-0.245816,-0.081701,0,-0.112988,-0.09651,1.226181,-0.146045,-0.290869,-0.26026,-0.691011,1,0,1,0,-0.5,-0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,1.535092,-0.903659,-0.236415,0.267240,-0.245816,-0.081701,0,-0.112988,-0.09651,-0.336686,-0.146045,0.170125,-0.26026,-0.691011,1,0,1,0,-0.5,-0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,

In [129]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

predictions = model.predict(X_test_scaled)
probabilities = model.predict_proba(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, predictions):.2%}")

Accuracy: 78.61%
